# Phase 2: GRPO Alignment on Colab 🧠

**Optimized for free Colab T4 GPU (15GB VRAM)**

This notebook loads the SFT adapter from Phase 1, defines reward functions,
and runs GRPO (Group Relative Policy Optimization) to align the model.

Key optimizations:
- Unsloth FastLanguageModel (2x faster, 70% less memory)
- VLLM Standby mode for memory-efficient RL rollouts
- `num_generations=4` for meaningful GRPO advantage signal
- Real code execution reward (runs unit tests)
- Checkpoint saving (survives Colab disconnects)

In [ ]:
%%capture
# Install Unsloth + all dependencies (clean single install)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl" peft accelerate bitsandbytes huggingface_hub
# Upgrade datasets separately — Colab default version is too old (missing Json feature type)
!pip install -q -U datasets

In [ ]:
# Must be set BEFORE importing unsloth — reduces VRAM during RL rollouts
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 📦 Load Dataset

For GRPO, we need the `prompt` field (model generates completions),
plus `unit_tests` and `answer` columns so reward functions can execute and verify code.

In [ ]:
from datasets import load_dataset
import json as json_module

dataset = load_dataset("myounes21/logos-reasoning-dataset", split="train")

def format_grpo(example):
    """Format for GRPO: add prompt field, keep all other columns."""
    prompt = f"<|im_start|>user\n{example['instruction']}<|im_end|>\n<|im_start|>assistant\n"

    # Serialize unit_tests to JSON string (GRPOTrainer needs string columns)
    ut = example.get('unit_tests', [])
    ut_str = json_module.dumps(ut, ensure_ascii=False) if ut else '[]'

    return {"prompt": prompt, "unit_tests_json": ut_str}

train_dataset = dataset.map(format_grpo)
print(f"✅ Loaded {len(train_dataset)} prompts for GRPO training.")

# Verify unit_tests column is present
has_tests = sum(1 for x in train_dataset if x["unit_tests_json"] != "[]")
print(f"📊 {has_tests} examples have unit tests for code execution reward.")
print(f"\nSample prompt:\n{train_dataset[0]['prompt'][:300]}")

## 🎯 Reward Functions

Four automated judges score each generated completion:

| Reward | Max Score | What it checks |
|--------|-----------|----------------|
| Format | +1.0 | Uses `<think>` tags and ` ```python ` code blocks |
| Correctness | +5.0 | Extracted code **passes unit tests** (or has `def` if no tests) |
| Language | +2.0 | Reasoning is in Arabic (contains Arabic characters) |
| Logic | +1.0 | Uses Arabic logical connectors (2 keywords = max) |

In [ ]:
import re
import json as json_module
import signal
import traceback

# ── Helper: Extract python code from ```python ... ``` blocks ──
def extract_code_block(text):
    """Extract code from the last ```python ... ``` block in the completion."""
    matches = re.findall(r'```python\n(.*?)```', text, re.DOTALL)
    return matches[-1].strip() if matches else None

# ── Helper: Execute code and run unit tests with timeout ──
def run_unit_tests(code, unit_tests, timeout=3):
    """Execute code, find the main function, run unit tests against it.
    Returns score from 0.0 to 5.0 based on fraction of tests passed.
    """
    if not code or not unit_tests:
        return None  # Can't evaluate

    # Find the main function name (last top-level def, skip __init__ etc.)
    func_names = [f for f in re.findall(r'^def (\w+)\(', code, re.MULTILINE)
                  if not f.startswith('__')]
    if not func_names:
        return 0.0  # No function found
    func_name = func_names[-1]

    # Execute the code in a sandboxed namespace
    namespace = {}
    try:
        exec(code, namespace)
    except Exception:
        return 0.0  # Code doesn't even compile/run

    func = namespace.get(func_name)
    if not callable(func):
        return 0.0

    # Run each unit test
    passed = 0
    total = len(unit_tests)

    for test in unit_tests:
        inp = test.get('input', '')
        expected = test.get('expected', '')
        try:
            # Build the function call string and evaluate it
            call_str = f'{func_name}({inp})'
            # Use alarm for timeout (Linux/Colab only)
            signal.alarm(timeout)
            result = eval(call_str, namespace)
            signal.alarm(0)  # Cancel alarm

            # Compare result to expected (handle string vs native types)
            if isinstance(expected, str):
                try:
                    expected_val = eval(expected)
                except:
                    expected_val = expected
            else:
                expected_val = expected

            if result == expected_val:
                passed += 1
        except Exception:
            signal.alarm(0)  # Cancel alarm on error
            continue

    return 5.0 * (passed / total) if total > 0 else 0.0

# ── Reward 1: Format ──
def format_reward(completions, **kwargs):
    """Rewards correct output format: <think> tags + python code blocks."""
    scores = []
    for c in completions:
        has_think = "<think>" in c and "</think>" in c
        has_code = "```python" in c
        scores.append(1.0 if (has_think and has_code) else 0.0)
    return scores

# ── Reward 2: Correctness (real code execution!) ──
def correctness_reward(completions, unit_tests_json=None, **kwargs):
    """Rewards completions where extracted code passes unit tests.
    Falls back to checking for 'def ' if no unit tests available.
    """
    scores = []
    for i, c in enumerate(completions):
        code = extract_code_block(c)

        # Parse unit tests from JSON string
        tests = []
        if unit_tests_json and i < len(unit_tests_json):
            try:
                tests = json_module.loads(unit_tests_json[i]) if isinstance(unit_tests_json[i], str) else unit_tests_json[i]
            except:
                tests = []

        if tests and code:
            # Real execution: run unit tests
            score = run_unit_tests(code, tests)
            scores.append(score if score is not None else 0.0)
        elif code and 'def ' in code:
            # Fallback: at least has a function definition in the code block
            scores.append(2.0)
        else:
            scores.append(0.0)
    return scores

# ── Reward 3: Arabic Language ──
def language_reward(completions, **kwargs):
    """Rewards Arabic language usage in the reasoning trace."""
    return [2.0 if re.search(r"[\u0600-\u06FF]", c) else 0.0 for c in completions]

# ── Reward 4: Arabic Logic Keywords ──
def logic_reward(completions, **kwargs):
    """Rewards use of Arabic logical connectors indicating structured thought."""
    keywords = ["إذن", "بالتالي", "لأن", "بما أن", "نستنتج", "أولاً", "ثانياً", "أخيراً"]
    scores = []
    for c in completions:
        count = sum(1 for kw in keywords if kw in c)
        scores.append(min(1.0, count * 0.5))  # 2 keywords = max reward
    return scores

# ── Sanity test ──
test_completion = ["<think>\nإذن نحتاج بالتالي\n</think>\n```python\ndef solve(n):\n    return n * 2\n```"]
test_unit_tests = ['[{"input": "5", "expected": 10}]']

print("Format:", format_reward(test_completion))
print("Correctness:", correctness_reward(test_completion, unit_tests_json=test_unit_tests))
print("Language:", language_reward(test_completion))
print("Logic:", logic_reward(test_completion))

## 🏗️ Load SFT Adapter as Base Model

**Critical:** We load the SFT adapter *directly* as the base model, then apply a new LoRA on top.
This ensures GRPO training starts from the SFT checkpoint, not from scratch.

In [ ]:
from unsloth import FastLanguageModel
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

max_seq_length = 2048

# ⚠️ IMPORTANT: Load SFT adapter directly as the base model
# This ensures GRPO training builds ON TOP of SFT, not from scratch
SFT_ADAPTER_PATH = "/content/drive/MyDrive/logos-sft-adapter-unsloth-final"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_ADAPTER_PATH,  # Load SFT adapter directly
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Apply NEW LoRA on top of SFT for GRPO training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print(f"✅ SFT adapter loaded from: {SFT_ADAPTER_PATH}")
print(f"✅ New GRPO LoRA applied on top.")
!nvidia-smi

## 🏋️ GRPO Training

**T4 constraints:**
- `num_generations=4` for meaningful GRPO advantage signal
- `max_completion_length=512` (reduced from 768 to fit 4 generations in VRAM)
- Checkpoints saved every 25 steps to Google Drive

**⚠️ If you get OOM:** reduce `num_generations` to 2 and increase `max_completion_length` back to 768.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers.trainer_utils import get_last_checkpoint

gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR = "/content/drive/MyDrive/logos-grpo-adapter"

grpo_config = GRPOConfig(
    output_dir = OUTPUT_DIR,
    learning_rate = 1e-5,
    beta = 0.1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,    # Effective batch size = 4
    num_generations = 4,                # 4 completions per prompt for GRPO advantage
    max_prompt_length = 512,
    max_completion_length = 512,        # Reduced from 768 to fit 4 generations
    num_train_epochs = 1,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",              # 8-bit optimizer to save VRAM
    logging_steps = 5,
    save_strategy = "steps",
    save_steps = 25,
    save_total_limit = 3,
    report_to = "none",
    seed = 3407,
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [format_reward, correctness_reward, language_reward, logic_reward],
    args = grpo_config,
    train_dataset = train_dataset,
)

# Resume automatically if checkpoint exists
checkpoint = get_last_checkpoint(OUTPUT_DIR)

print("🧠 Starting GRPO training...")
if checkpoint is not None:
    print(f"🔄 Resuming from checkpoint: {checkpoint}")
    trainer.train(resume_from_checkpoint=checkpoint)
else:
    print("🆕 No checkpoint found. Starting fresh training.")
    trainer.train()

# Save final adapter
ADAPTER_DIR = "/content/drive/MyDrive/logos-grpo-adapter-final"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\n✅ GRPO training complete! Adapter saved to: {ADAPTER_DIR}")

## 📤 Push to HuggingFace Hub (Optional)

In [ ]:
# Optional: Login to HuggingFace
from huggingface_hub import login
login()  # This will prompt for your token

# Push GRPO adapter to Hub
HF_REPO = "myounes21/logos-grpo-adapter"  # Change this to your repo name
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"✅ GRPO adapter pushed to: https://huggingface.co/{HF_REPO}")